# 01 — Export magic-window pool12 data (Shalow)

Pipeline step 1: export sequences from **2013–2022** (same noon anchors / K=2 crops as `01_export_random_window.ipynb`), but spatial crops come from the **12 fixed windows** in `Magic_window/window_selection.ipynb`.

**Settings**
- `DT = 360`, 3 cond + 1 pred, u+v
- **Pool:** noon-only (`HOUR_OF_DAY = 12`), **K=2 crops per anchor** → up to **~8000 rows**
- **Window pool:** **12 catalog crops per row** (slot 0..11 = export keys below), cached sims → `{prefix}_sims_pool_k1.npy`
- **Training:** each epoch randomly picks 1 of 12 slots (per-window loss can use slot index later)
- **Mask filter:** drop if land/coastal invalid **≥ 5%** (`MAX_MASK_FRAC = 0.05`)
- **Test:** 2023 noon full year, **1 crop per anchor**, same 12-window pool
- **Device:** `CUDA_VISIBLE_DEVICES=7` (export workers stay on CPU)

Catalog source: `Magic_window/window_catalog.py` (`catalog_id = magic_window_12`).


In [ ]:
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
SHALOW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Shalow" else NOTEBOOK_DIR / "Shalow"
ROOT = SHALOW_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SHALOW_DIR) not in sys.path:
    sys.path.insert(0, str(SHALOW_DIR))

DATA_DIR = SHALOW_DIR / "data"
LOG_DIR = SHALOW_DIR / "logs"
for _d in (DATA_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.export.fix_window_export import candidate_indices_for_hour
from src.export.random_window_export import (
    RandomWindowExportConfig,
    export_wants_fresh_start,
    plan_multi_crop_export,
    run_export,
)
from window_paths import export_manifest_path, pool_prefix, test_prefix, window_data_dir
from Magic_window.window_catalog import CATALOG_ID, catalog_manifest, get_resolved_catalog


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

ZARR_PATH = os.getenv(
    "GUIANA_ZARR_PATH",
    "s3://oceangen-data/margem_equatorial_tensor.zarr",
)

EXPORT_KEY = "magic_pool12_128"
CATALOG_ID = CATALOG_ID  # magic_window_12 from window_selection.ipynb
WINDOW_MODE = "catalog"
CROP_HEIGHT = 128
CROP_WIDTH = 128

DT = 360
CONDITIONING_STEPS = 3
FORECAST_STEPS = 1
EXPORT_COMPONENTS = ("uv",)

WINDOW_DATA_DIR = window_data_dir(DATA_DIR, EXPORT_KEY)
POOL_DATA_PREFIX_BASE = pool_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
TEST_DATA_PREFIX_BASE = test_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
EXPORT_MANIFEST_PATH = export_manifest_path(DATA_DIR, EXPORT_KEY)


def _component_export_settings(component: str, base: str | None = None):
    if component == "uv":
        base = base or POOL_DATA_PREFIX_BASE
        return base, ("u", "v"), (0, 1)
    raise ValueError(f"Unknown component {component!r}; use 'uv'")


PREVIEW_COMPONENT = EXPORT_COMPONENTS[0]
OUTPUT_PREFIX, CHANNEL_NAMES, TARGET_CHANNEL_INDICES = _component_export_settings(PREVIEW_COMPONENT)

SEED = 42
N_JOBS = 30
MAX_MASK_FRAC = 0.05
MAX_CROP_ATTEMPTS = 200
HOUR_OF_DAY = 12
CROPS_PER_ANCHOR = 2

EXPORT_WINDOW_POOL = True
N_WINDOWS_PER_SAMPLE = 12
TRAIN_CROPS_FOR_WINDOW_POOL = (1, 2)
EXPORT_EPOCH_WINDOWS = False

N_EXPORT_EPOCHS = 100
RANDOM_CROP_SEED = 42
VAL_SPLIT = 0.3
VAL_SPLIT_MODE = "random"
VAL_SPLIT_SEED = 42
TRAIN_CROPS_FOR_EPOCH_WINDOWS = (1, 2)

N_SAMPLES_TO_GEN = 8000
START_TIME = "2013-01-01"
END_TIME = "2022-12-31"

CATALOG = get_resolved_catalog(CATALOG_ID)
assert len(CATALOG) == N_WINDOWS_PER_SAMPLE

WINDOW_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
base_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
    start_time=START_TIME,
    end_time=END_TIME,
)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Catalog:", CATALOG_ID, "|", len(CATALOG), "windows from window_selection.ipynb")
for spec in CATALOG:
    print(
        f"  slot {spec['slot']:2d}: {spec['export_key']:<28} "
        f"top={spec['window_top']:3d} left={spec['window_left']:3d} angle={spec['window_angle']:6.1f}°"
    )

print("Opening Zarr:", ZARR_PATH)
dataset = GuianaDataloader(**base_kwargs, transform=None)
info = dataset.info()
candidate_idx = candidate_indices_for_hour(dataset, HOUR_OF_DAY)

pool_anchors, pool_zarr_idx, pool_crop_slots = plan_multi_crop_export(
    candidate_idx,
    n_samples_to_gen=N_SAMPLES_TO_GEN,
    crops_per_anchor=CROPS_PER_ANCHOR,
    seed=SEED,
)
n_anchors = len(pool_anchors)
n_rows = len(pool_zarr_idx)

export_manifest = {
    "export_key": EXPORT_KEY,
    "window_data_dir": str(WINDOW_DATA_DIR),
    "window_mode": WINDOW_MODE,
    "catalog_id": CATALOG_ID,
    "catalog": catalog_manifest(CATALOG),
    "mode": "catalog_pool_export",
    "sample_seed": SEED,
    "n_pool": int(len(candidate_idx)),
    "n_anchors": int(n_anchors),
    "crops_per_anchor": int(CROPS_PER_ANCHOR),
    "n_rows": int(n_rows),
    "start_time": START_TIME,
    "end_time": END_TIME,
    "hour_of_day": HOUR_OF_DAY,
    "crop_height": CROP_HEIGHT,
    "crop_width": CROP_WIDTH,
    "max_mask_frac": MAX_MASK_FRAC,
    "pool_data_prefix_base": POOL_DATA_PREFIX_BASE,
    "train_val_split_note": "70/30 split at training time",
    "export_window_pool": EXPORT_WINDOW_POOL,
    "n_windows_per_sample": N_WINDOWS_PER_SAMPLE,
    "train_crops_for_window_pool": list(TRAIN_CROPS_FOR_WINDOW_POOL),
    "random_crop_seed": RANDOM_CROP_SEED,
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "zarr_indices": pool_anchors.tolist(),
}
EXPORT_MANIFEST_PATH.write_text(json.dumps(export_manifest, indent=2))

print("Valid samples:", len(dataset))
print("After hour filter:", len(candidate_idx), "| HOUR_OF_DAY =", HOUR_OF_DAY)
print(
    f"Pool plan: {n_anchors} noon anchors × {CROPS_PER_ANCHOR} crops = {n_rows} export rows "
    f"(seed={SEED}) → {POOL_DATA_PREFIX_BASE}"
)
print("Export manifest:", EXPORT_MANIFEST_PATH)
print("Grid (H, W):", info["space"]["height"], info["space"]["width"])
print(
    f"Window pool: {N_WINDOWS_PER_SAMPLE} catalog crops/row, k={TRAIN_CROPS_FOR_WINDOW_POOL} "
    f"(slot 0..11 = fixed export keys; random slot each epoch at train time)"
)
if n_rows < N_SAMPLES_TO_GEN:
    print(
        f"NOTE: {n_rows} rows < N_SAMPLES_TO_GEN={N_SAMPLES_TO_GEN} "
        f"(limited by {len(candidate_idx)} noon anchors × {CROPS_PER_ANCHOR})"
    )


## Parallel export (resumable)

One export job for the 2013–2022 pool. Re-run or use the CLI to resume.

After the pool merge finishes, **catalog window pool** export runs automatically (`_sims_pool_k1.npy`).

```bash
CUDA_VISIBLE_DEVICES=7 python scripts/export_random_window_data.py --output-prefix <POOL_PREFIX> --window-pool-only
```


In [ ]:
import subprocess

DETACHED = True
CHUNK_SIZE = 100
RESUME = True


def _run_one_export(
    output_prefix: str,
    channel_names: tuple[str, ...],
    target_channel_indices: tuple[int, ...],
    *,
    start_time: str | None = None,
    end_time: str | None = None,
    n_samples_to_gen: int | None = None,
    crops_per_anchor: int | None = None,
    fixed_dataset_indices: tuple[int, ...] | None = None,
):
    export_config = RandomWindowExportConfig(
        zarr_path=ZARR_PATH,
        dt=DT,
        conditioning_steps=CONDITIONING_STEPS,
        forecast_steps=FORECAST_STEPS,
        crop_height=CROP_HEIGHT,
        crop_width=CROP_WIDTH,
        seed=SEED,
        n_jobs=N_JOBS,
        max_mask_frac=MAX_MASK_FRAC,
        max_crop_attempts=MAX_CROP_ATTEMPTS,
        hour_of_day=HOUR_OF_DAY,
        start_time=start_time or START_TIME,
        end_time=end_time or END_TIME,
        output_prefix=output_prefix,
        n_samples_to_gen=n_samples_to_gen if n_samples_to_gen is not None else N_SAMPLES_TO_GEN,
        crops_per_anchor=crops_per_anchor if crops_per_anchor is not None else CROPS_PER_ANCHOR,
        chunk_size=CHUNK_SIZE,
        channel_names=channel_names,
        target_channel_indices=target_channel_indices,
        fixed_dataset_indices=fixed_dataset_indices,
        export_key=EXPORT_KEY,
        export_epoch_windows=EXPORT_EPOCH_WINDOWS,
        export_window_pool=EXPORT_WINDOW_POOL,
        n_windows_per_sample=N_WINDOWS_PER_SAMPLE,
        train_crops_for_window_pool=TRAIN_CROPS_FOR_WINDOW_POOL,
        n_export_epochs=N_EXPORT_EPOCHS,
        random_crop_seed=RANDOM_CROP_SEED,
        val_split=VAL_SPLIT,
        val_split_mode=VAL_SPLIT_MODE,
        val_split_seed=VAL_SPLIT_SEED,
        train_crops_for_epoch_windows=TRAIN_CROPS_FOR_EPOCH_WINDOWS,
        window_mode=WINDOW_MODE,
        catalog_id=CATALOG_ID,
    )

    config_path = Path(f"{output_prefix}_export_config.json")
    config_dict = asdict(export_config)
    config_dict["window_mode"] = WINDOW_MODE
    config_path.write_text(json.dumps(config_dict, indent=2, default=str))
    print(f"Wrote config: {config_path}")

    fresh_start = export_wants_fresh_start(export_config, resume=RESUME)

    if DETACHED:
        log_path = LOG_DIR / f"export_{EXPORT_KEY}_{Path(output_prefix).name}.log"
        cmd = [
            sys.executable,
            str(ROOT / "scripts/export_random_window_data.py"),
            "--output-prefix",
            output_prefix,
        ]
        if fresh_start:
            cmd.append("--no-resume")
            if log_path.exists():
                log_path.unlink()
                print(f"Cleared log: {log_path}")
        log_mode = "w" if fresh_start else "a"
        export_env = os.environ.copy()
        export_env["CUDA_VISIBLE_DEVICES"] = "7"
        with open(log_path, log_mode) as log_file:
            proc = subprocess.Popen(
                cmd,
                stdout=log_file,
                stderr=subprocess.STDOUT,
                start_new_session=True,
                cwd=ROOT,
                env=export_env,
            )
        print(f"Started background export PID {proc.pid} -> {output_prefix}")
        if fresh_start:
            print("Fresh export (log cleared, partial state reset if config changed)")
        else:
            print(f"Resuming export (appending to {log_path})")
        print(f"Log: {log_path}")
        print(f"Monitor: tail -f {log_path}")
        print(
            f"Resume: python scripts/export_random_window_data.py --output-prefix {output_prefix}"
        )
    else:
        run_export(export_config, resume=not fresh_start)


for component in EXPORT_COMPONENTS:
    pool_pfx, ch_names, tgt_idx = _component_export_settings(component, POOL_DATA_PREFIX_BASE)
    print(
        f"\n=== Pool export: {component} -> {pool_pfx} "
        f"({n_anchors} anchors × {CROPS_PER_ANCHOR} = {n_rows} rows) ==="
    )
    _run_one_export(
        pool_pfx,
        ch_names,
        tgt_idx,
        n_samples_to_gen=n_rows,
        crops_per_anchor=CROPS_PER_ANCHOR,
        fixed_dataset_indices=tuple(int(x) for x in pool_anchors),
    )


## 2023 test export (rollout holdout)


In [ ]:
TEST_START_TIME = "2023-01-01"
TEST_END_TIME = "2023-12-31"
TEST_EXPORT_ALL = True

test_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
    start_time=TEST_START_TIME,
    end_time=TEST_END_TIME,
)
test_dataset = GuianaDataloader(**test_kwargs, transform=None)
test_candidate_idx = candidate_indices_for_hour(test_dataset, HOUR_OF_DAY)
print(f"2023 test pool: {len(test_candidate_idx)} starts | HOUR_OF_DAY={HOUR_OF_DAY}")
test_n = len(test_candidate_idx) if TEST_EXPORT_ALL else min(N_SAMPLES_TO_GEN, len(test_candidate_idx))
print(f"Will export: {test_n} test sequences")

for component in EXPORT_COMPONENTS:
    test_pfx, test_ch_names, test_tgt_idx = _component_export_settings(
        component, base=TEST_DATA_PREFIX_BASE
    )
    print(f"\n=== 2023 test export: {component} -> {test_pfx} ===")
    _run_one_export(
        test_pfx,
        test_ch_names,
        test_tgt_idx,
        start_time=TEST_START_TIME,
        end_time=TEST_END_TIME,
        n_samples_to_gen=test_n,
        crops_per_anchor=1,
        fixed_dataset_indices=tuple(int(x) for x in test_candidate_idx) if TEST_EXPORT_ALL else None,
    )
